In [1]:
import os, sys
from pathlib import Path

# Localizar a raiz do projeto (pasta que contém "src/") para poder importar os módulos auxiliares
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
print("Raiz do projeto:", PROJECT_ROOT)

Raiz do projeto: /mnt/attach/outputs/IA3_Projeto-Creditcard


# 3. Modeling (Metodologia CRISP-ML)

## 3.1. Visão geral e estratégia de treino
Comparam-se algoritmos de **Machine Learning (IA I e IA II)** e **Deep Learning (TensorFlow/Keras)** para classificar fraude.

1. **Divisão temporal**: 80% mais antigo = treino; 20% mais recente = teste (sem shuffle).
2. **Desequilíbrio**: `class_weight` + métricas da classe fraude (Precision, Recall, F1, PR-AUC).
3. **Persistência**: modelos otimizados (`*_opt.joblib`, `*.keras`), scaler e features em `models/`.

## 3.2. Baseline (Regressão Logística e Random Forest)

In [2]:
from src.train_model import treinar_baseline
tabela_baseline = treinar_baseline()
tabela_baseline

Regressão Logística
Matriz de confusão (linhas=real, colunas=previsto):
[[54186  2486]
 [    6    68]]


Random Forest
Matriz de confusão (linhas=real, colunas=previsto):
[[56671     1]
 [   20    54]]


,Modelo,Accuracy (%),Precision Fraude (%),Recall Fraude (%),F1 Fraude (%),ROC-AUC (%),PR-AUC (%),MCC
0,Regressão Logística,95.61,2.66,91.89,5.18,98.72,76.26,0.1523
1,Random Forest,99.96,98.18,72.97,83.72,97.07,81.16,0.8463


## 3.3. Otimização de hiperparâmetros via GridSearchCV (IA I & IA II)
`GridSearchCV` com **validação cruzada temporal** (`TimeSeriesSplit`) e métrica **Average Precision** (PR-AUC).
SVM, KNN e Gradient Boosting são otimizados numa **amostra** (todas as fraudes + N legítimas) por limitações de tempo/CPU,
mas avaliados no teste completo.

In [3]:
import pandas as pd
from src.tune_model import otimizar_modelos_ml
tabela_ml, scores_ml = otimizar_modelos_ml()
tabela_ml.drop(columns=["Melhores Hiperparâmetros"])

[Logistic Regression] CV AP=0.7590 | teste PR-AUC=75.92% | 5s | {'C': 0.1, 'class_weight': 'balanced'}


[SVM] CV AP=0.8993 | teste PR-AUC=69.06% | 12s | {'C': 10.0, 'class_weight': None, 'gamma': 'scale'}


[Random Forest] CV AP=0.7674 | teste PR-AUC=81.41% | 170s | {'class_weight': 'balanced_subsample', 'max_depth': 12, 'n_estimators': 100}


[Decision Tree] CV AP=0.6978 | teste PR-AUC=70.99% | 124s | {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 12, 'min_samples_leaf': 10}


[Gradient Boosting] CV AP=0.7509 | teste PR-AUC=38.34% | 336s | {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 150, 'subsample': 0.8}


[Naive Bayes] CV AP=0.0629 | teste PR-AUC=6.19% | 1s | {'var_smoothing': np.float64(0.001)}


[KNN] CV AP=0.8546 | teste PR-AUC=79.33% | 69s | {'metric': 'manhattan', 'n_neighbors': 11, 'weights': 'distance'}


,Modelo,Tipo,Ficheiro,Escalado,Treino,Accuracy (%),Precision Fraude (%),Recall Fraude (%),F1 Fraude (%),ROC-AUC (%),PR-AUC (%),MCC
0,Logistic Regression,Machine Learning,logistic_regression_opt.joblib,True,completo (226980),95.63,2.68,91.89,5.20,98.71,75.92,0.1528
1,SVM,Machine Learning,svm_opt.joblib,True,amostra (15399),99.88,52.43,72.97,61.02,96.70,69.06,0.6180
2,Random Forest,Machine Learning,random_forest_opt.joblib,False,completo (226980),99.96,98.15,71.62,82.81,98.62,81.41,0.8383
3,Decision Tree,Machine Learning,decision_tree_opt.joblib,False,completo (226980),99.17,11.11,77.03,19.42,88.41,70.99,0.2906
4,Gradient Boosting,Machine Learning,gradient_boosting_opt.joblib,False,amostra (40399),99.89,55.21,71.62,62.35,98.42,38.34,0.6283
5,Naive Bayes,Machine Learning,naive_bayes_opt.joblib,True,completo (226980),98.01,5.03,79.73,9.46,96.79,6.19,0.1972
6,KNN,Machine Learning,knn_opt.joblib,True,amostra (40399),99.94,76.71,75.68,76.19,93.15,79.33,0.7616


## 3.4. Treino do modelo de Deep Learning em TensorFlow / Keras
MLP (Dense 64→32→16, *Dropout*, saída *sigmoid*), `class_weight` e *EarlyStopping* sobre a PR-AUC de validação.

In [4]:
from src.tune_model import treinar_tensorflow
tabela_tf, score_tf = treinar_tensorflow(verbose=False)
tabela_tf.drop(columns=["Melhores Hiperparâmetros"])

I0000 00:00:1794825344.133027     295 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1794825345.265586     295 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


,Modelo,Tipo,Ficheiro,Escalado,Treino,Accuracy (%),Precision Fraude (%),Recall Fraude (%),F1 Fraude (%),ROC-AUC (%),PR-AUC (%),MCC
0,TensorFlow Neural Net,Deep Learning,tensorflow_mlp_opt.keras,True,completo (226980),99.1,11.23,85.14,19.84,95.49,79.4,0.3073


## 3.5. Quadro comparativo da modelação

In [5]:
tabela = pd.concat([tabela_ml, tabela_tf], ignore_index=True).sort_values("PR-AUC (%)", ascending=False)
tabela.drop(columns=["Melhores Hiperparâmetros", "Ficheiro", "Escalado"]).reset_index(drop=True)

,Modelo,Tipo,Treino,Accuracy (%),Precision Fraude (%),Recall Fraude (%),F1 Fraude (%),ROC-AUC (%),PR-AUC (%),MCC
0,Random Forest,Machine Learning,completo (226980),99.96,98.15,71.62,82.81,98.62,81.41,0.8383
1,TensorFlow Neural Net,Deep Learning,completo (226980),99.10,11.23,85.14,19.84,95.49,79.40,0.3073
2,KNN,Machine Learning,amostra (40399),99.94,76.71,75.68,76.19,93.15,79.33,0.7616
3,Logistic Regression,Machine Learning,completo (226980),95.63,2.68,91.89,5.20,98.71,75.92,0.1528
4,Decision Tree,Machine Learning,completo (226980),99.17,11.11,77.03,19.42,88.41,70.99,0.2906
5,SVM,Machine Learning,amostra (15399),99.88,52.43,72.97,61.02,96.70,69.06,0.6180
6,Gradient Boosting,Machine Learning,amostra (40399),99.89,55.21,71.62,62.35,98.42,38.34,0.6283
7,Naive Bayes,Machine Learning,completo (226980),98.01,5.03,79.73,9.46,96.79,6.19,0.1972


In [6]:
for _, linha in tabela.iterrows():
    print(f"{linha['Modelo']:<24} -> {linha['Melhores Hiperparâmetros']}")

Random Forest            -> {'class_weight': 'balanced_subsample', 'max_depth': 12, 'n_estimators': 100}
TensorFlow Neural Net    -> Dense(64->32->16), Dropout(0.3/0.2), Adam(lr=0.0008), sigmoid, class_weight
KNN                      -> {'metric': 'manhattan', 'n_neighbors': 11, 'weights': 'distance'}
Logistic Regression      -> {'C': 0.1, 'class_weight': 'balanced'}
Decision Tree            -> {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 12, 'min_samples_leaf': 10}
SVM                      -> {'C': 10.0, 'class_weight': None, 'gamma': 'scale'}
Gradient Boosting        -> {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 150, 'subsample': 0.8}
Naive Bayes              -> {'var_smoothing': np.float64(0.001)}
